# Gradient Descent Demo

To demonstrate Gradient Descent, we will minimize the function $f(x,y)=0.25x^4-x^2+.2x+.5y^2$. In this case, we have two parameters, $x$ and $y$.

Now, this has no inherent ML meaning, but the key thing is, if we change $x$ or $y$, the value of the output will change - and we're trying to find the values of $x$ and $y$ that minimize it.  First, here's what the loss function looks like:

In [1]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(2) # For reproducibility


def loss_function(x, y):
    """
    Calculates the loss for a given x and y.
    Based on the 'Peaks' function.
    """
    return 0.25 * x**4 - x**2 + 0.20 * x + 0.50 * y**2

# 1. Setup the Grid
x_range = np.linspace(-3, 3, 100)
y_range = np.linspace(-3, 3, 100)
X, Y = np.meshgrid(x_range, y_range)

# 2. Calculate Loss (Z)
Z = loss_function(X, Y)

# 3. Plotting with Plotly
fig = go.Figure(data=[go.Surface(z=Z, x=X, y=Y, colorscale='Viridis', opacity=0.9, showscale=False)])

fig.update_layout(
    title='Non-Convex Loss Surface',
    autosize=True,
    width=None,
    height=800,
    margin=dict(l=0, r=0, b=0, t=40), # remove whitespace margins
    scene=dict(
        aspectmode='cube', # Forces X, Y, and Z to have equal lengths
        xaxis=dict(nticks=10, range=[-3, 3]),
        yaxis=dict(nticks=10, range=[-3, 3]),
        zaxis=dict(nticks=10, range=[-2, 6]), # Adjust Z range if needed
    )
)

fig.show()

You'll notice it has local minima, global minima, and is very nonconvex. So, we have no choice but to do gradient descent. To start, we create a random guess for $x$ and $y$.  That random guess is visible in the below plot as a red diamond.

In [2]:
# 1. Generate a random initial guess
# We restrict the range slightly to ensuring the point is clearly visible
start_x = np.random.uniform(-2.5, 2.5)
start_y = np.random.uniform(-2.5, 2.5)
start_z = loss_function(start_x, start_y)

print(f"Initial Guess: x={start_x:.4f}, y={start_y:.4f}, Loss={start_z:.4f}")

# 2. Add the point to the existing plot
# This uses the 'fig' object created in the previous cell
fig.add_trace(
    go.Scatter3d(
        x=[start_x],
        y=[start_y],
        z=[start_z],
        mode='markers',
        marker=dict(
            size=8,
            color='red',
            symbol='diamond',
            opacity=1.0
        ),
        name='Initial Guess'
    )
)

# 3. Render the updated figure
fig.show()

Initial Guess: x=-0.3200, y=-2.3704, Loss=2.6455


We now calculate the gradient $\nabla_{x,y} f(x,y)$, which is to say, $\frac{\partial f(x,y)}{\partial x}$ and $\frac{\partial f(x,y)}{\partial y}$. They're not hard!

$$\frac{\partial f(x,y)}{\partial x} = x^3-2x+.2$$

$$\frac{\partial f(x,y)}{\partial y}=y$$

That makes a vector pointing in the direction that most steeply uphill from the estimate. We want to move in the opposite direction. The red "cone" points in the direction we should step.

In [3]:
def calculate_gradient(x, y):
    """
    Calculates the gradient of the tilted double-well potential.
    f(x, y) = 0.25*x^4 - x^2 + 0.20*x + 0.50*y^2
    """
    # Partial derivative with respect to x
    dx = x**3 - 2*x + 0.20
    
    # Partial derivative with respect to y
    dy = y
    
    return dx, dy

def overlay_gradient_arrow(fig, x, y, visual_length=0.75):
    """
    Adds a normalized 3D cone representing the negative gradient 
    to the existing Plotly figure.
    
    visual_length: The approximate length of the arrow in axis units. 
                   Since axes are -3 to 3, 0.75 is a reasonable default.
    """
    # 1. Calculate raw gradient vectors
    grad_x, grad_y = calculate_gradient(x, y)
    z_val = loss_function(x, y)

    # Descent direction in XY plane
    u_raw = -grad_x 
    v_raw = -grad_y 
    # Change in Z along that direction
    w_raw = -(grad_x**2 + grad_y**2)

    # 2. Normalize vector to fix visual size
    # Calculate magnitude of the 3D vector
    magnitude = np.sqrt(u_raw**2 + v_raw**2 + w_raw**2)
    
    # Avoid division by zero at local extrema
    if magnitude < 1e-9:
        print(f"Gradient is near zero at ({x:.3f}, {y:.3f}). Skipping arrow.")
        return

    # Scale components to the desired visual length
    u = (u_raw / magnitude) * visual_length
    v = (v_raw / magnitude) * visual_length
    w = (w_raw / magnitude) * visual_length

    # 3. Add the Cone trace with absolute sizing
    fig.add_trace(
        go.Cone(
            x=[x], y=[y], z=[z_val],    # Anchor point
            u=[u], v=[v], w=[w],        # Normalized vector components
            sizemode="absolute",        # Use the absolute values of u,v,w for size
            sizeref=1,                  # Reference factor of 1
            anchor="tail",              # Anchor the cone at the tail
            showscale=False,
            colorscale=[[0, 'rgb(255,0,0)'], [1, 'rgb(255,0,0)']], # Solid red
            name='Gradient Vector'
        )
    )
    
    print(f"Gradient at ({x:.3f}, {y:.3f}): <{grad_x:.3f}, {grad_y:.3f}>")

# Clear previous gradient traces if re-running this cell repeatedly
# to avoid cluttered plot.
fig.data = [trace for trace in fig.data if trace.name not in ['Gradient Vector']]

# Add the new, normalized arrow
overlay_gradient_arrow(fig, start_x, start_y)
fig.show()

Gradient at (-0.320, -2.370): <0.807, -2.370>


We can then subtract this gradient, giving us a new estimate. We repeat over and over again until we find a minimum.

In [4]:
def run_gradient_descent(start_x, start_y, learning_rate=0.05, n_iterations=50):
    """
    Performs standard Gradient Descent.
    Returns lists of x, y, and z coordinates for the path.
    """
    # Initialize history with starting point
    path_x = [start_x]
    path_y = [start_y]
    path_z = [loss_function(start_x, start_y)]

    curr_x, curr_y = start_x, start_y

    for i in range(n_iterations):
        # 1. Calculate Gradient
        grad_x, grad_y = calculate_gradient(curr_x, curr_y)

        # 2. Update Parameters (Descent step)
        curr_x = curr_x - learning_rate * grad_x
        curr_y = curr_y - learning_rate * grad_y

        # 3. Store new position
        path_x.append(curr_x)
        path_y.append(curr_y)
        path_z.append(loss_function(curr_x, curr_y))

    return path_x, path_y, path_z

# --- Execution ---

# Parameters
lr = .05
steps = 50

# Run the optimization
px, py, pz = run_gradient_descent(start_x, start_y, learning_rate=lr, n_iterations=steps)

# --- Visualization ---

# 1. Add the Path Trajectory
fig.add_trace(
    go.Scatter3d(
        x=px,
        y=py,
        z=pz,
        mode='lines+markers',
        marker=dict(
            size=4,
            color='white',
            symbol='circle',
            opacity=0.8
        ),
        line=dict(
            color='white',
            width=5
        ),
        name='GD Trajectory'
    )
)

# 2. Highlight the Final Position
fig.add_trace(
    go.Scatter3d(
        x=[px[-1]],
        y=[py[-1]],
        z=[pz[-1]],
        mode='markers',
        marker=dict(
            size=10,
            color='yellow',
            symbol='x'
        ),
        name='Final Estimate'
    )
)

print(f"Start: ({px[0]:.3f}, {py[0]:.3f}) -> Loss: {pz[0]:.3f}")
print(f"End:   ({px[-1]:.3f}, {py[-1]:.3f}) -> Loss: {pz[-1]:.3f}")

fig.show()

Start: (-0.320, -2.370) -> Loss: 2.646
End:   (-1.462, -0.182) -> Loss: -1.271
